In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import logging
import os
import random
import time
import re
import math
import numpy as np
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

# --- Selenium ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

# --- Requests ---
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter('ignore', InsecureRequestWarning)

# ===============================================================
#  全域設定
# ===============================================================

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_FILENAME = os.path.join(BASE_DIR, 'news_dataset_v5.csv')
LOG_FILENAME = os.path.join(BASE_DIR, 'scraper_v9_5.log')
V5_COLUMNS = ['url', 'media', 'category', 'publish_time', 'title', 'content', 'fetch_time', 'label']

# 強制目標數量
LIMIT_L1 = 50
LIMIT_L0 = 75

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILENAME, encoding='utf-8', mode='a'),
        logging.StreamHandler(sys.stdout)
    ]
)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Referer': 'https://www.google.com/'
}

# ===============================================================
#  [模組 1] 選擇器與行為模擬 (維持 V9.4 優點)
# ===============================================================

GENERIC_LIST_SELECTORS = [
    "h3 a", "div.news-list-group a", "div.news_list li a", 
    "ul.list li a.tit", "div.story-list__text h2 a", 
    "section.article-list li h3 a", "div.block_content h3 a", 
    "div.white-box a", "li a"
]

GENERIC_CONTENT_SELECTORS = [
    'div.article-content', 'div.story', 'div.article-body', 
    'div[itemprop="articleBody"]', 'section.article-content__editor',
    'div.news_content', 'article', 'div.text', 'div.p-note',
    'div.fncnews-content', 'div.post-content', 'div.m-content'
]

class HumanNavigator:
    def __init__(self, driver):
        self.driver = driver
        self.action = ActionChains(driver)

    def random_sleep(self, min_s=1.5, max_s=3.5):
        time.sleep(random.uniform(min_s, max_s))

    def smooth_scroll(self):
        try:
            current_pos = self.driver.execute_script("return window.pageYOffset;")
            target_distance = random.randint(600, 1000)
            target_pos = current_pos + target_distance
            steps = random.randint(10, 20)
            for _ in range(steps):
                move = (target_distance / steps) + random.uniform(-10, 10) 
                current_pos += move
                self.driver.execute_script(f"window.scrollTo(0, {current_pos});")
                time.sleep(random.uniform(0.02, 0.1))
        except: pass

    def random_mouse_move(self):
        try:
            links = self.driver.find_elements(By.TAG_NAME, "a")
            if len(links) > 0:
                target = random.choice(links[:20])
                self.action.move_to_element(target).perform()
        except: pass 

    def simulate_reading_behavior(self):
        self.smooth_scroll()
        self.random_mouse_move()
        self.random_sleep(1.0, 2.0)

# ===============================================================
#  [模組 2] 資料解析工具
# ===============================================================

def format_time_user_style(time_str):
    if not time_str or str(time_str).lower() in ['nan', 'n/a', 'none']:
        dt = datetime.now()
    else:
        try:
            clean = str(time_str).replace("發布時間：", "").replace("更新時間：", "").strip()
            dt = date_parser.parse(clean, fuzzy=True)
        except:
            dt = datetime.now()
    
    hour_12 = int(dt.strftime("%I"))
    time_part = f"{hour_12}:{dt.strftime('%M:%S %p')}"
    return f"{dt.year}/{dt.month}/{dt.day} {time_part}"

def parse_html_content(html_text):
    soup = BeautifulSoup(html_text, 'lxml')
    content = np.nan
    for sel in GENERIC_CONTENT_SELECTORS:
        elm = soup.select_one(sel)
        if elm:
            for tag in elm(['script', 'style', 'iframe', 'div.knn_related', 'div.fb-root', 'div.author', 'div.ad', 'div.click_mic']):
                tag.decompose()
            text = elm.get_text(separator=' ', strip=True)
            text = re.sub(r'\s+', ' ', text)
            if len(text) > 20:
                content = text
                break
    
    category = "即時"
    meta_sec = soup.select_one('meta[property="article:section"]')
    if meta_sec and meta_sec.get('content'):
        category = meta_sec.get('content')
    else:
        crumbs = soup.select('div.breadcrumb a, nav a, div.guide a')
        if len(crumbs) >= 2:
            txt = crumbs[1].get_text(strip=True)
            if len(txt) < 8: category = txt

    raw_time = None
    time_elm = soup.select_one('meta[property="article:published_time"], meta[name="pubdate"], time, span.time, div.date')
    if time_elm:
        raw_time = time_elm.get('content') if time_elm.has_attr('content') else time_elm.get_text(strip=True)
            
    return category, format_time_user_style(raw_time), content

# ===============================================================
#  [模組 3] 爬蟲核心 (NewsSpider)
# ===============================================================

class NewsSpider:
    def __init__(self):
        self.options = Options()
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--window-size=1920,1080")
        self.options.add_argument(f"user-agent={HEADERS['User-Agent']}")
        self.driver = None
        self.human = None
        self.existing_urls = self._get_existing_urls()

    def _get_existing_urls(self):
        if os.path.exists(DATA_FILENAME):
            try:
                df = pd.read_csv(DATA_FILENAME)
                if 'url' in df.columns: return set(df['url'].tolist())
            except: pass
        return set()

    def start(self):
        """啟動瀏覽器"""
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.human = HumanNavigator(self.driver)
        # logging.info("   🔧 WebDriver 已啟動 (Session Start)")

    def stop(self):
        """徹底關閉瀏覽器，釋放記憶體"""
        if self.driver: 
            try:
                self.driver.quit()
                # logging.info("   🔧 WebDriver 已關閉 (Session Kill)")
            except: pass
            self.driver = None

    def fetch_content_smart(self, url):
        # 1. Try Requests
        try:
            res = requests.get(url, headers=HEADERS, timeout=6, verify=False)
            if res.status_code == 200:
                cat, t, c = parse_html_content(res.text)
                if isinstance(c, str): return cat, t, c
        except: pass

        # 2. Try Selenium (Fallback)
        try:
            self.driver.execute_script("window.open('');")
            self.driver.switch_to.window(self.driver.window_handles[-1])
            self.driver.get(url)
            self.human.random_sleep(1.5, 2.5) 
            html = self.driver.page_source
            self.driver.close()
            self.driver.switch_to.window(self.driver.window_handles[0])
            return parse_html_content(html)
        except:
            try:
                if len(self.driver.window_handles) > 1:
                    self.driver.close()
                    self.driver.switch_to.window(self.driver.window_handles[0])
            except: pass
        
        return "即時", format_time_user_style(None), np.nan

    def crawl_site(self, name, url, label, limit):
        logging.info(f"      L{label} 任務啟動 (目標: {limit} 筆)...")
        try:
            self.driver.get(url)
            self.human.random_sleep(2, 4)

            collected = []
            seen_links = set()
            
            # 1. 捲動掃描
            max_scroll = int((limit / 8) * 1.8) + 3
            scroll_count = 0
            
            while scroll_count < max_scroll:
                self.human.simulate_reading_behavior()
                scroll_count += 1
                
                if scroll_count % 2 == 0:
                    soup = BeautifulSoup(self.driver.page_source, 'lxml')
                    links_found = 0
                    for sel in GENERIC_LIST_SELECTORS:
                        found = soup.select(sel)
                        if found: links_found += len(found)
                    
                    if links_found >= limit * 1.5: break
            
            # 2. 解析
            soup = BeautifulSoup(self.driver.page_source, 'lxml')
            elements = []
            for sel in GENERIC_LIST_SELECTORS:
                found = soup.select(sel)
                if found: elements.extend(found)

            unique_elements = []
            _temp_seen = set()
            for el in elements:
                h = el.get('href')
                if h and h not in _temp_seen:
                    _temp_seen.add(h)
                    unique_elements.append(el)
            
            # 3. 抓取
            for item in unique_elements:
                if len(collected) >= limit: break
                
                try:
                    link = item.get('href')
                    title = item.get('title') or item.get_text(strip=True)
                    if not link or not title or len(title) < 4: continue
                    
                    if link.startswith('/'):
                        if 'setn' in url: base = 'https://www.setn.com'
                        elif 'ltn' in url: base = 'https://news.ltn.com.tw'
                        elif 'tvbs' in url: base = 'https://news.tvbs.com.tw'
                        elif 'chinatimes' in url: base = 'https://www.chinatimes.com'
                        elif 'ettoday' in url: base = 'https://www.ettoday.net'
                        elif 'udn' in url: base = 'https://udn.com'
                        else: base = ""
                        full_url = base + link
                    elif link.startswith('http'):
                        full_url = link
                    else: continue
                    
                    clean_url = full_url.split('?')[0]
                    if clean_url in seen_links: continue
                    seen_links.add(clean_url)

                    is_exist = False
                    for exist in self.existing_urls:
                        if clean_url in str(exist):
                            is_exist = True; break
                    
                    if label == 0 and is_exist: continue 
                    
                    cat, t_str, content = self.fetch_content_smart(full_url)
                    fetch_time = format_time_user_style(str(datetime.now()))
                    
                    collected.append({
                        'url': full_url,
                        'media': name,
                        'category': cat,
                        'publish_time': t_str,
                        'title': title,
                        'content': content,
                        'fetch_time': fetch_time,
                        'label': label
                    })
                    
                    status = "OK" if isinstance(content, str) else "NaN"
                    print(f"         [L{label}] {title[:8]}... ({status})")
                    time.sleep(random.uniform(0.5, 1.2))

                except Exception: continue
            
            return collected
        except Exception as e:
            logging.error(f"      🔥 錯誤: {e}")
            return []

# ===============================================================
#  [設定] 媒體清單整合 (方便一站式處理)
# ===============================================================

# 將熱門與即時整合成一個 Config，方便迴圈處理
MEDIA_CONFIGS = [
    {
        'name': 'SETN',
        'hot': 'https://www.setn.com/ViewAll.aspx?PageGroupID=6',
        'latest': 'https://www.setn.com/ViewAll.aspx'
    },
    {
        'name': 'TVBS',
        'hot': 'https://news.tvbs.com.tw/hot',
        'latest': 'https://news.tvbs.com.tw/realtime'
    },
    {
        'name': 'ETtoday',
        'hot': 'https://www.ettoday.net/news/hot-news.htm',
        'latest': 'https://www.ettoday.net/news/news-list.htm'
    },
    {
        'name': 'LTN',
        'hot': 'https://news.ltn.com.tw/list/breakingnews/popular',
        'latest': 'https://news.ltn.com.tw/list/breakingnews'
    },
    {
        'name': 'ChinaTimes',
        'hot': 'https://www.chinatimes.com/hotnews/?chdtv',
        'latest': 'https://www.chinatimes.com/realtimenews/?chdtv'
    },
    {
        'name': 'UDN',
        'hot': 'https://udn.com/rank/pv/2',
        'latest': 'https://udn.com/news/breaknews/1'
    }
]

def main():
    logging.info(f"🚀 T-Forecast V9.5 (Anti-Zombie & Memory Safe) 啟動")
    logging.info(f"🎯 目標: 熱門(L1)={LIMIT_L1}, 即時(L0)={LIMIT_L0}")

    spider = NewsSpider()
    
    # 迴圈遍歷每一家媒體
    for media in MEDIA_CONFIGS:
        site_name = media['name']
        logging.info(f"--- 處理媒體: {site_name} ---")
        
        # 1. 啟動瀏覽器 (全新 Session)
        spider.start()
        
        all_data = [] # 暫存這家媒體的資料
        
        try:
            # 2. 抓熱門 (L1)
            l1_data = spider.crawl_site(site_name, media['hot'], label=1, limit=LIMIT_L1)
            all_data.extend(l1_data)
            logging.info(f"      ✅ {site_name} L1 完成: {len(l1_data)} 筆")
            
            # 中場休息
            spider.human.random_sleep(2, 3)
            
            # 3. 抓即時 (L0)
            l0_data = spider.crawl_site(site_name, media['latest'], label=0, limit=LIMIT_L0)
            all_data.extend(l0_data)
            logging.info(f"      ✅ {site_name} L0 完成: {len(l0_data)} 筆")
            
        except Exception as e:
            logging.error(f"   🔥 {site_name} 發生錯誤: {e}")
        
        # 4. 關閉瀏覽器 (關鍵步驟：防止殭屍化)
        spider.stop()
        logging.info(f"   💤 瀏覽器已重置，休息 5 秒...")
        time.sleep(5)

        # 5. 存檔 (每抓完一家就存一次，降低風險)
        if all_data:
            df = pd.DataFrame(all_data)
            for col in V5_COLUMNS:
                if col not in df.columns: df[col] = np.nan
            df = df[V5_COLUMNS]
            try:
                df.to_csv(DATA_FILENAME, mode='a', header=False, index=False, encoding='utf-8-sig')
                logging.info(f"   💾 {site_name} 資料存檔成功 (共 {len(all_data)} 筆)")
            except:
                df.to_csv(f"backup_{site_name}_{int(time.time())}.csv", index=False)

    logging.info("🏁 所有媒體任務結束")

if __name__ == "__main__":
    main()

2025-12-08 22:28:09,183 - INFO - 🚀 T-Forecast V9.5 (Anti-Zombie & Memory Safe) 啟動
2025-12-08 22:28:09,183 - INFO - 🎯 目標: 熱門(L1)=50, 即時(L0)=75
2025-12-08 22:28:09,197 - INFO - --- 處理媒體: SETN ---
2025-12-08 22:28:09,198 - INFO - ====== WebDriver manager ======
2025-12-08 22:28:09,454 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-08 22:28:09,704 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-08 22:28:09,906 - INFO - Driver [/Users/xiaobing/.wdm/drivers/chromedriver/mac64/142.0.7444.175/chromedriver-mac-arm64/chromedriver] found in cache
2025-12-08 22:28:10,494 - INFO -       L1 任務啟動 (目標: 50 筆)...
         [L1] 美國防授權法案納... (OK)
         [L1] APP下載... (NaN)
         [L1] AI爆報... (NaN)
         [L1] 追蹤新聞... (NaN)
         [L1] 台灣彩券開獎直播... (OK)
         [L1] 三立iNEWS新... (OK)
         [L1] LIVE三立+2... (OK)
         [L1] 小紅書遭封鎖... (NaN)
         [L1] iNEWS... (NaN)
         [L1] 三立官網... (NaN)
         [L1] 關於三立... (NaN)
         [L1] 關於三立新聞網... (NaN)
   